# 03 · Evaluation protocol: baselines and the shipped v2 network

Scores every model on the same fixed temporal blocks, with the same metrics, using `citibike_eval.py`.
The protocol is described in `docs/evaluation.md` and does not change between experiments.

**Inputs:** `data_procesada/` from `LIMPIEZA_citibike.ipynb` (12 months, Oct 2025 to Sep 2026) and the shipped model `ui/models/demanda_net_v2.onnx`.
**Outputs:** CSV files in `data/results/`.

| Block | Dates | Use |
| --- | --- | --- |
| Rolling folds 1 to 5 | Aug 13 to Sep 16, one week each | Choose settings, train on everything before the fold |
| Validation | Sep 3 to 16 | Choose the baseline to beat |
| Test | Sep 17 to 30 | Final score only. It never drives a decision |

> The test block is scored once per frozen candidate. Nothing in this notebook is tuned on it.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

import citibike_eval as ce

REPO = Path('..')
DATA_DIR = Path('data_procesada')
RESULTS = REPO / 'data' / 'results'
CACHE = REPO / 'tmp' / 'cache'
V2_ONNX = REPO / 'ui' / 'models' / 'demanda_net_v2.onnx'
V2_IDS = REPO / 'ui' / 'models' / 'station_ids.json'
RESULTS.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

np.random.seed(ce.SEED)   # nothing here is random; the seed is fixed for the notebooks that build on it

def save(df, name):
  df.round(6).to_csv(RESULTS / name, index=False)
  print(f'saved {RESULTS / name} ({len(df):,} rows)')

## 1. Data
The dense arrays are cached in `tmp/cache/`, so reloading is instant.

In [2]:
data = ce.load_processed(DATA_DIR, cache_dir=CACHE)
print(f'{data.n_est:,} stations | {data.n_days} days | {data.days[0]:%Y-%m-%d} to {data.days[-1]:%Y-%m-%d}')
print(f'workdays {int((data.daytype == 0).sum())} | weekend or holiday {int((data.daytype == 1).sum())} | federal holidays {int(data.holiday.sum())}')
blocks = [data.block(n, a, b) for n, a, b in ce.FOLDS] + [data.block('val', *ce.VAL), data.block('test', *ce.TEST)]
pd.DataFrame([dict(block=b.name, first=data.days[b.d0].date(), last=data.days[b.d1 - 1].date(), days=b.d1 - b.d0) for b in blocks])

2,515 stations | 365 days | 2025-10-01 to 2026-09-30
workdays 250 | weekend or holiday 115 | federal holidays 12


,block,first,last,days
0,fold1,2026-08-13,2026-08-19,7
1,fold2,2026-08-20,2026-08-26,7
2,fold3,2026-08-27,2026-09-02,7
3,fold4,2026-09-03,2026-09-09,7
4,fold5,2026-09-10,2026-09-16,7
5,val,2026-09-03,2026-09-16,14
6,test,2026-09-17,2026-09-30,14


## 2. Baselines and the shipped v2 network

Baselines are historical means with a cold-start rule (a station with no training data in a group gets the median station profile of that group).

| Name | Definition |
| --- | --- |
| `A` | Station mean |
| `B` | Station x weekday x slot of the day |
| `C` | Station x workday-or-free x slot of the day |
| `all`, `8w`, `jul-sep` | Training window: whole history before the block, the last 8 weeks, or Jul 1 to Sep 2 (the window v2 was trained on) |
| `C|all+level` | `C|all` rescaled by the city-wide level of the last 4 weeks |

`train-only` means the fit ends on Sep 2.
`refit` means it ends on Sep 16 (train plus validation) and is only reported for the test block.
v2 is the shipped checkpoint, trained on Jul 1 to Sep 2 and early-stopped on the validation block, so its validation row is not fully held out.

In [3]:
t0 = time.time()
base, base_test = ce.run_baselines(data)
v2, v2_test = ce.run_v2(data, V2_ONNX, V2_IDS)
metrics = pd.concat([base, v2], ignore_index=True)
test_preds = {**base_test, **v2_test}
print(f'{len(metrics):,} metric rows in {time.time() - t0:.0f}s')
save(metrics, 'point_metrics.csv')

972 metric rows in 147s
saved ../data/results/point_metrics.csv (972 rows)


## 3. The baseline to beat
Chosen on the validation block by 15-minute Poisson deviance, never on the test block.

In [4]:
bar, bar_table = ce.choose_bar(metrics)
print('Baseline to beat:', bar)
save(bar_table.reset_index(), 'bar_selection_val.csv')
bar_table.round(4)

Baseline to beat: C|all+level
saved ../data/results/bar_selection_val.csv (8 rows)


,deviance_15min,mae_15min,wape_hour,wape_day,bias
model,,,,,
C|all+level,0.9344,0.5784,0.4640,0.2135,0.9124
C|all,1.0391,0.5822,0.5266,0.3715,0.6675
C|jul-sep,1.0599,0.5849,0.4711,0.2202,0.9036
C|8w,1.0883,0.5886,0.4725,0.2193,0.9293
B|all,1.1899,0.5834,0.5277,0.3699,0.6719
A|all,1.4852,0.7523,0.7849,0.3830,0.6718
B|jul-sep,1.6654,0.5962,0.4827,0.2230,0.9036
B|8w,1.7666,0.6009,0.4846,0.2190,0.9282


## 4. Test block, Sep 17 to 30
Mean of the departures-total and arrivals-total targets.
`deviance_15min` is the Poisson deviance per station and 15 minutes (primary).
`wape_hour` and `wape_day` are the sum of absolute errors over the sum of actuals per station-hour and station-day (primary).
`bias` is the sum of predictions over the sum of actuals.
MAE is shown for completeness, it rewards under-prediction when most counts are zero.

In [5]:
test = metrics[metrics['block'] == 'test']
rows = []
for fit in ['train-only', 'train-only (shipped)', 'refit']:
  s = ce.summary(test[test['fit'] == fit])
  s.insert(0, 'fit', fit)
  rows.append(s)
table = pd.concat(rows).reset_index().rename(columns={'index': 'model'})
order = ['v2 network', 'C|all+level', 'C|all', 'C|8w', 'C|jul-sep', 'B|all', 'A|all']
table = table[table['model'].isin(order)].assign(rank=lambda d: d['model'].map(order.index)).sort_values(['fit', 'rank']).drop(columns='rank')
table.round(4).reset_index(drop=True)

,model,fit,deviance_15min,mae_15min,wape_hour,wape_day,bias
0,C|all+level,refit,0.8993,0.5677,0.5675,0.3310,1.0968
1,C|all,refit,0.9246,0.5391,0.5694,0.3979,0.7981
2,C|8w,refit,1.0204,0.5812,0.5881,0.3500,1.0990
3,B|all,refit,1.0174,0.5362,0.5634,0.3918,0.7943
4,A|all,refit,1.3596,0.7019,0.8538,0.4306,0.7944
5,C|all+level,train-only,0.9111,0.5658,0.5674,0.3339,1.0693
6,C|all,train-only,0.9447,0.5409,0.5758,0.4077,0.7824
7,C|8w,train-only,1.0648,0.5864,0.6003,0.3634,1.0772
8,C|jul-sep,train-only,1.0362,0.5788,0.5908,0.3572,1.0504
9,B|all,train-only,1.0541,0.5388,0.5712,0.4034,0.7789


In [6]:
t = ce.summary(test[test['fit'].isin(['train-only', 'train-only (shipped)'])])
gain = (1 - t.loc['v2 network'] / t.loc[bar]).rename('v2 vs bar (lower is better)')
print(f'Improvement of the shipped v2 network over {bar} on the test block:')
print((gain * 100).round(1).drop('bias').astype(str) + ' %')

Improvement of the shipped v2 network over C|all+level on the test block:
deviance_15min     8.9 %
mae_15min          8.8 %
wape_hour         12.3 %
wape_day          23.5 %
Name: v2 vs bar (lower is better), dtype: str


## 5. Rolling origin
Mean over the five weekly folds, each fitted on everything before it.

In [7]:
roll = metrics[metrics['fit'] == 'rolling']
ce.summary(roll).loc[['C|all+level', 'C|all', 'C|8w', 'B|all', 'B|8w', 'A|all']].round(4)

,deviance_15min,mae_15min,wape_hour,wape_day,bias
model,,,,,
C|all+level,0.9363,0.5763,0.4948,0.2338,0.9913
C|all,1.0142,0.5635,0.5301,0.3487,0.7089
C|8w,1.0700,0.5752,0.4871,0.2159,0.9765
B|all,1.1519,0.5627,0.5280,0.3489,0.7085
B|8w,1.7157,0.5853,0.4953,0.2149,0.9753
A|all,1.4146,0.7228,0.7877,0.3573,0.7086


In [8]:
per_fold = (roll[(roll['target'] == 'salidas_total') & (roll['level'] == '15min')]
            .pivot_table(index='model', columns='block', values='deviance').loc[['C|all+level', 'C|all', 'C|8w']])
per_fold.round(4)

block,fold1,fold2,fold3,fold4,fold5
model,,,,,
C|all+level,0.9275,0.9258,0.9112,0.9008,0.9194
C|all,1.0241,0.9910,0.9359,0.9816,1.0438
C|8w,1.0532,1.0407,1.0054,1.0519,1.0523


## 6. Breakdowns on the test block
By station volume tier (departures per day in the training window), borough, neighborhood (NTA), hour, weekday and day type.
Stations with no training data are their own group.
Neighborhoods come from `ui/src/data/regions.json`, and stations missing there are placed by point-in-polygon.

In [9]:
blk = data.block('test', *ce.TEST)
Yb, actb, slb = ce.block_arrays(data, blk)
groups = ce.station_groups(data, 0, data.day(ce.TRAIN_END) + 1, REPO)
chosen = {'v2 network': test_preds[('v2 network', 'train-only (shipped)')],
          f'{bar} (bar)': test_preds[(bar, 'train-only')],
          'C|all': test_preds[('C|all', 'train-only')]}
parts = []
for name, P in chosen.items():
  for target in ['salidas_total', 'llegadas_total']:
    b = ce.breakdown(Yb, P, actb, slb, groups, target=target)
    b.insert(0, 'model', name)
    parts.append(b)
breakdown = pd.concat(parts, ignore_index=True)
save(breakdown, 'breakdown_test.csv')

saved ../data/results/breakdown_test.csv (990 rows)


In [10]:
def show(dimension, target='salidas_total', metrics=('deviance', 'wape', 'bias')):
  b = breakdown[(breakdown['dimension'] == dimension) & (breakdown['target'] == target)]
  return b.pivot_table(index='group', columns='model', values=list(metrics)).round(3)

tiers = ['no training data', '<2', '2-5', '5-15', '15-40', '>40']
show('volume_tier').loc[tiers]

bias                              deviance                                wape                             
model             C|all C|all+level (bar) v2 network    C|all C|all+level (bar) v2 network  C|all C|all+level (bar) v2 network
group                                                                                                                         
no training data  0.791             1.083      2.512    1.285             1.273      1.528  1.599             1.821      2.877
<2                0.594             0.812      6.132    0.150             0.150      0.174  1.534             1.732      7.062
2-5               0.772             1.058      0.906    0.401             0.400      0.283  1.647             1.887      1.761
5-15              0.762             1.046      0.889    0.545             0.538      0.491  1.499             1.685      1.576
15-40             0.776             1.063      0.888    0.855             0.837      0.804  1.211             1.299      1.205
>40               0.785             1.074      0.880    1.494             1.420      1.285  0.696             0.714      0.639

In [11]:
show('borough')

bias                              deviance                                wape                             
model       C|all C|all+level (bar) v2 network    C|all C|all+level (bar) v2 network  C|all C|all+level (bar) v2 network
group                                                                                                                   
Bronx       0.733             1.011      0.868    0.546             0.534      0.518  1.389             1.540      1.466
Brooklyn    0.801             1.096      0.928    0.896             0.878      0.788  0.886             0.939      0.866
Manhattan   0.777             1.062      0.869    1.403             1.324      1.195  0.689             0.705      0.633
Queens      0.794             1.085      0.907    0.639             0.629      0.586  1.171             1.272      1.179
unassigned    NaN               NaN        NaN    0.000             0.000      0.348    NaN               NaN        NaN

In [12]:
show('day_type')

bias                              deviance                                wape                             
model    C|all C|all+level (bar) v2 network    C|all C|all+level (bar) v2 network  C|all C|all+level (bar) v2 network
group                                                                                                                
weekend  1.125              1.54      0.944    0.907             0.991      0.737  1.146             1.344      0.965
workday  0.709              0.97      0.877    0.944             0.863      0.855  0.728             0.730      0.723

In [13]:
show('hour', metrics=('deviance', 'bias'))

bias                              deviance                             
model  C|all C|all+level (bar) v2 network    C|all C|all+level (bar) v2 network
group                                                                          
0      0.821             1.125      0.980    0.674             0.670      0.592
1      0.827             1.135      1.009    0.505             0.504      0.440
2      0.820             1.125      1.003    0.383             0.382      0.324
3      0.838             1.149      1.016    0.301             0.301      0.253
4      0.876             1.201      1.038    0.302             0.304      0.255
5      0.800             1.095      0.941    0.444             0.440      0.391
6      0.780             1.068      0.882    0.677             0.661      0.607
7      0.723             0.990      0.698    0.951             0.884      0.908
8      0.749             1.024      0.808    1.121             1.039      0.976
9      0.802             1.096      0.892    1.044             1.014      0.906
10     0.868             1.187      0.838    0.986             0.994      0.909
11     0.869             1.188      0.870    1.048             1.057      0.936
12     0.831             1.136      0.908    1.085             1.073      0.984
13     0.820             1.121      0.935    1.135             1.116      1.049
14     0.820             1.122      0.919    1.221             1.201      1.071
15     0.791             1.082      0.890    1.275             1.230      1.098
16     0.768             1.051      0.862    1.325             1.252      1.150
17     0.728             0.997      0.862    1.452             1.308      1.249
18     0.730             0.999      0.897    1.400             1.267      1.152
19     0.766             1.048      0.949    1.209             1.140      1.051
20     0.797             1.090      0.962    1.061             1.031      0.947
21     0.779             1.066      0.931    0.994             0.962      0.890
22     0.784             1.074      0.915    0.967             0.944      0.846
23     0.817             1.120      0.945    0.833             0.825      0.728

In [14]:
show('weekday', metrics=('deviance', 'wape', 'bias'))

bias                              deviance                                wape                             
model  C|all C|all+level (bar) v2 network    C|all C|all+level (bar) v2 network  C|all C|all+level (bar) v2 network
group                                                                                                              
0      0.937             1.282      1.039    0.795             0.832      0.753  0.820             0.918      0.819
1      0.705             0.965      0.901    0.912             0.829      0.833  0.714             0.708      0.714
2      0.664             0.908      0.877    0.968             0.855      0.858  0.706             0.684      0.696
3      0.651             0.891      0.787    0.991             0.868      0.908  0.704             0.679      0.698
4      0.657             0.899      0.831    1.052             0.933      0.922  0.723             0.713      0.714
5      0.822             1.124      0.777    1.006             0.994      0.834  0.927             1.004      0.814
6      1.785             2.443      1.309    0.808             0.989      0.640  1.622             2.084      1.293

In [15]:
nta = breakdown[(breakdown['dimension'] == 'nta') & (breakdown['target'] == 'salidas_total') & (breakdown['model'] == f'{bar} (bar)')]
print('Largest neighborhoods by departures, bar model:')
nta.nlargest(8, 'mean_y').assign(departures_per_cell=lambda d: d['mean_y'])[['group', 'n', 'deviance', 'wape', 'bias']].round(3).reset_index(drop=True)

Largest neighborhoods by departures, bar model:


,group,n,deviance,wape,bias
0,Chelsea-Hudson Yards,45792,1.848,0.565,0.964
1,Midtown South-Flatiron-Union Square,32256,1.532,0.539,1.041
2,Greenwich Village,26880,1.642,0.583,0.985
3,SoHo-Little Italy-Hudson Square,26880,1.613,0.619,1.042
4,West Village,29568,1.606,0.628,1.033
5,Hell's Kitchen,28224,1.698,0.665,1.126
6,Gramercy,13440,1.435,0.593,0.989
7,Upper West Side-Lincoln Square,22848,1.628,0.662,1.111


## 7. Probabilistic quality of the current assumption
Predictive distribution = Poisson with the model's rate, as the dashboard assumes.
The aggregated intervals use mu = sum of the slot rates, which assumes independent slots.
Coverage far below the nominal level at station-hour or station-day scale is the miscalibration that the negative binomial work must fix.
At 15 minutes most counts are 0, so a discrete interval cannot hit the nominal level and coverage is above it.

In [16]:
t0 = time.time()
iv = []
ls = []
for name in ['v2 network', f'{bar} (bar)']:
  P = chosen[name]
  i = ce.poisson_intervals(Yb, P, actb)
  i.insert(0, 'model', name)
  iv.append(i)
  s = ce.logscore_poisson(Yb, P, actb)
  ls.append(pd.DataFrame({'model': name, 'target': s.index, 'logscore': s.values}))
intervals = pd.concat(iv, ignore_index=True)
logscore = pd.concat(ls, ignore_index=True)
print(f'{time.time() - t0:.0f}s')
save(intervals, 'intervals_poisson_test.csv')
save(logscore, 'logscore_poisson_test.csv')
intervals[intervals['target'].isin(['salidas_total'])].pivot_table(index=['model', 'agg'], columns='level', values=['coverage', 'mean_width']).round(3)

13s
saved ../data/results/intervals_poisson_test.csv (72 rows)
saved ../data/results/logscore_poisson_test.csv (8 rows)


coverage        mean_width        
level                        0.8    0.9        0.8     0.9
model             agg                                     
C|all+level (bar) 15min    0.935  0.963      1.355   1.796
                  day      0.481  0.566     17.588  22.591
                  hour     0.857  0.905      3.096   3.956
v2 network        15min    0.932  0.960      1.178   1.587
                  day      0.530  0.621     15.840  20.347
                  hour     0.866  0.913      2.750   3.528

In [17]:
logscore.pivot_table(index='target', columns='model', values='logscore').round(4)

model,C|all+level (bar),v2 network
target,,
llegadas_classic,0.3735,0.3481
llegadas_electric,0.6938,0.6599
salidas_classic,0.3678,0.3443
salidas_electric,0.6900,0.6580


## 8. Reading the results
- The comparison that matters is the shipped v2 network against the strongest baseline chosen on validation.
- Level is the weak point of every model here: the test block is colder than most of the training history, see the `bias` column.
- Intervals under the Poisson assumption are far too narrow at station-day scale. This is the baseline for the interval work.
- Later models are appended to the same CSV files by `citibike_eval.py`, so every comparison uses the same blocks and metrics.